In [8]:
import pandas as pd
import numpy as np

vehicle_file = "Traffic_Crashes_-_Vehicles.csv"   # 改成你的路径

vehicle_keep_cols = [
    "CRASH_UNIT_ID",
    "CRASH_RECORD_ID",
    "CRASH_DATE",
    "UNIT_NO",
    "NUM_PASSENGERS",
    "UNIT_TYPE",
    "VEHICLE_ID",
    "CMRC_VEH_I",
    "VEHICLE_YEAR",
    "VEHICLE_DEFECT",
    "VEHICLE_TYPE",
    "VEHICLE_USE",
    "TRAVEL_DIRECTION",
    "MANEUVER",
    "TOWED_I",
    "FIRE_I",
    "OCCUPANT_CNT",
    "EXCEED_SPEED_LIMIT_I",
    "FIRST_CONTACT_POINT",
    "VEHICLE_CONFIG",
    "CARGO_BODY_TYPE",
    "LOAD_TYPE",
    "HAZMAT_CLASS"
]

vehicle = pd.read_csv(
    vehicle_file,
    usecols=lambda c: c in vehicle_keep_cols,
    low_memory=False
)

print("Original vehicle shape:", vehicle.shape)

vehicle.columns = vehicle.columns.str.lower()

vehicle = vehicle.dropna(subset=["crash_record_id", "crash_unit_id"])

vehicle["crash_date"] = pd.to_datetime(vehicle["crash_date"], errors="coerce")
vehicle = vehicle.dropna(subset=["crash_date"])

vehicle = vehicle[vehicle["crash_date"] >= pd.Timestamp("2018-01-01")].copy()

vehicle = vehicle.drop_duplicates(subset=["crash_unit_id"])

numeric_cols = [
    "unit_no",
    "num_passengers",
    "vehicle_id",
    "vehicle_year",
    "occupant_cnt"
]

for col in numeric_cols:
    vehicle[col] = pd.to_numeric(vehicle[col], errors="coerce")

vehicle.loc[(vehicle["vehicle_year"] < 1900) | (vehicle["vehicle_year"] > 2026), "vehicle_year"] = np.nan

cat_cols = [
    "unit_type",
    "cmrc_veh_i",
    "vehicle_defect",
    "vehicle_type",
    "vehicle_use",
    "travel_direction",
    "maneuver",
    "towed_i",
    "fire_i",
    "exceed_speed_limit_i",
    "first_contact_point",
    "vehicle_config",
    "cargo_body_type",
    "load_type",
    "hazmat_class"
]

for col in cat_cols:
    vehicle[col] = (
        vehicle[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

replace_map = {
    "": pd.NA,
    "UNKNOWN": pd.NA,
    "UNABLE TO DETERMINE": pd.NA,
    "NOT APPLICABLE": pd.NA
}

for col in cat_cols:
    vehicle[col] = vehicle[col].replace(replace_map)

yn_cols = [
    "cmrc_veh_i",
    "towed_i",
    "fire_i",
    "exceed_speed_limit_i"
]

for col in yn_cols:
    vehicle[col] = vehicle[col].map({"Y": 1, "N": 0}).astype("Int64")

def simplify_vehicle_type(x):
    if pd.isna(x):
        return "UNKNOWN"
    x = str(x)

    if "MOTORCYCLE" in x or "MOPED" in x or "MOTOR SCOOTER" in x:
        return "MOTORCYCLE"
    elif "PICKUP" in x or "VAN" in x or "SUV" in x or "PASSENGER" in x or "AUTOMOBILE" in x:
        return "PASSENGER_VEHICLE"
    elif "BUS" in x:
        return "BUS"
    elif "TRUCK" in x or "TRACTOR" in x or "SEMI" in x:
        return "TRUCK"
    elif "BICYCLE" in x or "PEDALCYCLE" in x:
        return "BICYCLE"
    elif "PEDESTRIAN" in x:
        return "PEDESTRIAN"
    else:
        return "OTHER"

vehicle["vehicle_category"] = vehicle["vehicle_type"].apply(simplify_vehicle_type)

fill_unknown_cols = [
    "vehicle_type",
    "vehicle_category",
    "vehicle_use",
    "maneuver",
    "vehicle_defect",
    "first_contact_point",
    "vehicle_config",
    "cargo_body_type",
    "load_type",
    "hazmat_class"
]

for col in fill_unknown_cols:
    vehicle[col] = vehicle[col].fillna("UNKNOWN")

weak_info_mask = (
    vehicle["vehicle_type"].eq("UNKNOWN") &
    vehicle["maneuver"].eq("UNKNOWN") &
    vehicle["vehicle_use"].eq("UNKNOWN")
)

vehicle = vehicle.loc[~weak_info_mask].copy()

final_cols = [
    "crash_unit_id",
    "crash_record_id",
    "crash_date",
    "unit_no",
    "num_passengers",
    "unit_type",
    "vehicle_id",
    "cmrc_veh_i",
    "vehicle_year",
    "vehicle_defect",
    "vehicle_type",
    "vehicle_category",
    "vehicle_use",
    "travel_direction",
    "maneuver",
    "towed_i",
    "fire_i",
    "occupant_cnt",
    "exceed_speed_limit_i",
    "first_contact_point",
    "vehicle_config",
    "cargo_body_type",
    "load_type",
    "hazmat_class"
]

vehicle_analysis = vehicle[final_cols].copy()

print("Analysis-ready vehicle shape:", vehicle_analysis.shape)
print(vehicle_analysis.head())


vehicle_analysis.to_csv("vehicle_cleaned.csv", index=False)
print("\nSaved: vehicle_cleaned.csv")

Original vehicle shape: (2107146, 23)
Analysis-ready vehicle shape: (1783023, 24)
   crash_unit_id                                    crash_record_id  \
5        1000001  57945047eb951f4152fde013708fa029caf5d33fc2a2c2...   
6        1000002  57945047eb951f4152fde013708fa029caf5d33fc2a2c2...   
7        1000003  7cf1e0da7413f35c258ad67a3f0d08b10c9cf94a59bb4e...   
8        1000004  7cf1e0da7413f35c258ad67a3f0d08b10c9cf94a59bb4e...   
9        1000005  7cf1e0da7413f35c258ad67a3f0d08b10c9cf94a59bb4e...   

           crash_date  unit_no  num_passengers unit_type  vehicle_id  \
5 2020-11-25 17:46:00        1             NaN    DRIVER    947651.0   
6 2020-11-25 17:46:00        2             1.0    DRIVER    947660.0   
7 2020-11-25 18:39:00        1             NaN    DRIVER    947657.0   
8 2020-11-25 18:39:00        2             NaN    PARKED    947665.0   
9 2020-11-25 18:39:00        3             NaN    DRIVER    947670.0   

   cmrc_veh_i  vehicle_year vehicle_defect  ...           

In [ ]:
# 计算每一列 UNKNOWN 或 NaN 比例
missing_ratio = (
    vehicle_analysis.isna().mean() +
    (vehicle_analysis == "UNKNOWN").mean()
)

missing_ratio = missing_ratio.sort_values(ascending=False)

print(missing_ratio.head(20))

# 删除 UNKNOWN/NaN 比例 > 0.9 的列
cols_to_drop = missing_ratio[missing_ratio > 0.9].index.tolist()

print("Dropping columns:", cols_to_drop)

vehicle_reduced = vehicle_analysis.drop(columns=cols_to_drop)

print("New shape:", vehicle_reduced.shape)

hazmat_class            0.999337
exceed_speed_limit_i    0.999228
fire_i                  0.999092
load_type               0.995334
cargo_body_type         0.991698
vehicle_config          0.991303
cmrc_veh_i              0.980112
towed_i                 0.864577
num_passengers          0.847577
vehicle_defect          0.480618
vehicle_year            0.157179
first_contact_point     0.096533
travel_direction        0.080705
unit_type               0.000008
occupant_cnt                 0.0
crash_unit_id                0.0
maneuver                     0.0
crash_record_id              0.0
vehicle_category             0.0
vehicle_type                 0.0
dtype: Float64
Dropping columns: ['hazmat_class', 'exceed_speed_limit_i', 'fire_i', 'load_type', 'cargo_body_type', 'vehicle_config', 'cmrc_veh_i']
New shape: (1783023, 17)


In [14]:
manual_drop = [
    "cmrc_veh_i",          # 商业标识，没用
    "towed_i",             # 很 sparse
    "fire_i",              # 极少发生
    "vehicle_config",      # 过细
    "cargo_body_type",
    "load_type",
    "hazmat_class",
    "vehicle_defect"       # 大量 UNKNOWN + 噪音
]

vehicle_reduced = vehicle_reduced.drop(
    columns=[c for c in manual_drop if c in vehicle_reduced.columns]
)

In [ ]:
import pandas as pd
import numpy as np
vehicle_file1 = "vehicle_cleaned.csv"
vehicle1 = pd.read_csv(
    vehicle_file1
)
vehicle1